In [3]:
import streamlit as st
import snowflake.connector
import pandas as pd


@st.cache_resource
def init_connection():
    return snowflake.connector.connect(
        user=st.secrets["snowflake"]["user"],
        password=st.secrets["snowflake"]["password"],
        account=st.secrets["snowflake"]["account"],
        warehouse=st.secrets["snowflake"]["warehouse"],
        database=st.secrets["snowflake"]["database"],
        schema=st.secrets["snowflake"]["schema"]
    )

# Query runner
@st.cache_data(ttl=600)
def run_query(query):
    conn = init_connection()
    cur = conn.cursor()
    cur.execute(query)
    df = cur.fetch_pandas_all()
    cur.close()
    return df


2025-08-16 16:51:36.550 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [ ]:
st.set_page_config(page_title="SQL Dashboard", layout="wide")
st.title("📊 SQL Queries Dashboard - Snowflake Data")

# Define your 15 queries here
queries = {
    "Query 1 - Providers by City": """
        select city, count(distinct name) as no_of_providers
        from providers
        group by city;
    """,

    "Query 2 - Receivers by City": """
        select city, count(distinct name) as no_of_receivers
        from receiver
        group by city;
    """,

    "Query 3 - Top Provider Type": """
        select provider_type, count(*) as no_of_contribution
        from foodlist
        group by provider_type
        order by no_of_contribution desc
        limit 1;
    """,

    "Query 4 - Providers in Tinamouth": """
        select address, contact
        from providers
        where city='Tinamouth';
    """,

    "Query 5 - Total Food Available": """
        select sum(quantity) as total_food_available
        from foodlist;
    """,

    "Query 6 - Top 2 Cities by Providers": """
        select p.city, count(*) as no_of_providers
        from providers p
        join foodlist f on p.provider_id=f.provider_id
        group by p.city
        order by no_of_providers desc
        limit 2;
    """,

    "Query 7 - Top 5 Foods Donated": """
        select food_name
        from foodlist
        group by food_name
        order by count(*) desc
        limit 5;
    """,

    "Query 8 - Claims by Food": """
        select food_name, count(*) as no_of_claims
        from foodlist
        group by food_name;
    """,

    "Query 9 - Claims Status Percentage": """
        select status, round((count(*) * 100/(select count(*) from claims)),2) as percentage
        from claims
        group by status;
    """,

    "Query 10 - Top Provider by Successful Claims": """
        select p.name, count(*) as no_of_successful_claims
        from providers p
        join foodlist f on p.provider_id=f.provider_id
        join claims c on c.food_id=f.food_id
        where c.status='Completed'
        group by p.name
        order by no_of_successful_claims desc
        limit 1;
    """,

    "Query 11 - Top Meal Type Donated": """
        select meal_type, count(*)
        from foodlist
        group by meal_type
        order by count(*) desc
        limit 1;
    """,

    "Query 12 - Food Types Distribution": """
        select food_type, count(*)
        from foodlist
        group by food_type
        order by count(*) desc;
    """,

    "Query 13 - Quantity Donated by Provider": """
        select p.name, sum(f.quantity) as total_quantity_of_food_donated
        from providers p
        join foodlist f on p.provider_id=f.provider_id
        group by p.name;
    """,

    "Query 14 - Avg Quantity Claimed by Receiver": """
        select r.name, round(avg(f.quantity),2) as average_qunatity_claimed
        from receiver r
        join claims c on r.receiver_id=c.receiver_id
        join foodlist f on f.food_id=c.food_id
        group by r.name;
    """
}

In [ ]:
# Sidebar for selecting query
st.sidebar.header("🔍 Select a Query")
selected_query = st.sidebar.selectbox("Choose a query to run:", list(queries.keys()))

# Run and display selected query
st.subheader(selected_query)
df = run_query(queries[selected_query])
st.dataframe(df, use_container_width=True)

# Optional: Show charts if numeric data exists
if len(df.columns) >= 2 and pd.api.types.is_numeric_dtype(df[df.columns[1]]):
    st.bar_chart(df.set_index(df.columns[0]))